# 【簡約版】GPT-2 推論エンジン入門

> 本記事は『GPT-2 推論エンジン入門』の簡約版です。各トピックの詳細な解説は本編をご参照ください。

https://zenn.dev/7shi/books/20260319-gpt2

# GPT-2 とは

GPT-2 は OpenAI が 2019 年に発表した言語モデルです。「テキストを受け取り、次のトークンを予測する」というシンプルな目的で訓練されています。

現代の LLM（GPT-4, Claude など）も Transformer アーキテクチャの基本はほぼ変わっておらず、GPT-2 との本質的な違いはパラメーター数と学習データの規模です。GPT-2 は現代の LLM の内部構造を理解するための最もコンパクトな出発点です。

なお、この記事で扱うのは**推論のみ**です。学習については対象外です。

# セットアップと実行

以下のリポジトリを使用します。

https://github.com/7shi/my-gpt2

```bash
git clone https://github.com/7shi/my-gpt2.git
cd my-gpt2
uv sync
make download
```

## テキスト生成

GPT-2 は「次に来る単語を予測する」ことだけを学習した Base モデルです。質問に答えたり指示に従ったりする訓練（Instruction Tuning）は受けていないため、入力したテキスト（プロンプト）の続きを確率的に生成することしかできません。

GPT-2 は主に英語のテキストで学習されたモデルです。プロンプト（書き出し）を与えて、続きを生成させてみましょう。

In [250]:
from my_gpt2.generate import generate

_ = generate("Once upon a time")

Once upon a time they were concerned that what was going on with their children might have carried over to some other world, but now that they had grown up there seems to


英語ベースの GPT-2 はバイトレベル BPE により日本語の入力を受け付けますが、日本語として意味の通じる文章は生成できません。日本語を生成するには、日本語で学習された `rinna/japanese-gpt2-small` を `-m` オプションで指定します。モデル構造は GPT-2 と同一で、トークナイザーのみ SentencePiece に変更されています。

In [254]:
# -n 20 -m rinna/japanese-gpt2-small に相当
_ = generate("吾輩は猫で", n_tokens_to_generate=20, model_id="rinna/japanese-gpt2-small")

吾輩は猫で言いたい放題 ~japanese cat from aria~ avex blog 吾輩


生成は確率的なため、実行するたびに異なる結果が得られます。以下のオプションで生成の振る舞いを調整できます。

- **Temperature** (`-t`): 低い値ほど確信度の高い単語が選ばれやすく、高い値ほど多様な出力
- **Top-k** (`-k`): 確率上位 k 個のトークンのみを候補にする
- **Top-p** (`-p`): 累積確率が p に達するまでのトークンを候補にする

# 推論パイプラインの全体像

GPT-2 に入力されたテキストは以下のパイプラインを通り、「次に来る単語の確率分布」として出力されます。

1. テキスト
   - トークナイザー
     - BPE
     - SentencePiece
2. トークン ID 列
   - Embedding
3. ベクトル列
   - Transformer Block × 12
     - LayerNorm
     - Attention
     - 残差接続
     - LayerNorm
     - MLP
     - 残差接続
   - 最終 LayerNorm
   - LM Head
4. ロジット
   - サンプリング
5. 次のトークン

## モデルの重み

safetensors 形式（約 523 MB）で格納されています。768 は埋め込み次元、50257 は語彙数、1024 は最大トークン列長です。

| キー | 形状 | 役割 |
|---|---|---|
| `wte.weight` | (50257, 768) | Embedding（トークン） |
| `wpe.weight` | (1024, 768) | Embedding（位置） |
| `h.0.ln_1.weight`, `.bias` | (768,) | LayerNorm（Attention 前） |
| `h.0.attn.c_attn.weight`, `.bias` | (768, 2304), (2304,) | Attention（Q,K,V 結合） |
| `h.0.attn.c_proj.weight`, `.bias` | (768, 768), (768,) | Attention（出力射影） |
| `h.0.ln_2.weight`, `.bias` | (768,) | LayerNorm（MLP 前） |
| `h.0.mlp.c_fc.weight`, `.bias` | (768, 3072), (3072,) | MLP（拡張） |
| `h.0.mlp.c_proj.weight`, `.bias` | (3072, 768), (768,) | MLP（射影） |
| `h.1` 〜 `h.11` | 同上 | Transformer Block × 12 |
| `ln_f.weight`, `.bias` | (768,) | 最終 LayerNorm |

## コア推論コード

推論パイプラインの中核は以下のステップに集約されます。

In [225]:
# 以降のコードを動かすための準備（このリポジトリの my_gpt2 を使用）
import math
import numpy as np
from my_gpt2.tokenizer import Tokenizer
from my_gpt2.loader import load_gpt2_weights
from my_gpt2.model import softmax, gelu

model_id = "openai-community/gpt2"
tokenizer = Tokenizer(model_id)
model = load_gpt2_weights(model_id)
wte, wpe, ln_f, blocks = model.wte, model.wpe, model.ln_f, model.blocks

In [226]:
text = "The capital of France is"

# Step 0: トークナイザー — テキストをトークンIDの列に変換
input_ids = tokenizer.encode(text)

print("text:", repr(text))
print("input_ids:", [(tokenizer.decode([input_id]), input_id) for input_id in input_ids])

text: 'The capital of France is'
input_ids: [('The', 464), (' capital', 3139), (' of', 286), (' France', 4881), (' is', 318)]


In [227]:
# Step 1: Embedding — トークンIDをベクトルに変換し、位置情報を加算
vectorized_input_ids = wte[input_ids]
indices = np.arange(len(input_ids))
positional_info = wpe[indices]
x = vectorized_input_ids + positional_info

print("vectorized_input_ids:", vectorized_input_ids)
print("indices:", indices)
print("positional_info:", positional_info)
print("x (embedded input):", x)

vectorized_input_ids: [[-0.06862289 -0.02029774  0.06447055 ...  0.06388663 -0.00998752
   0.00310699]
 [ 0.05727245 -0.00699701  0.09142003 ...  0.0372112  -0.02651583
   0.06396636]
 [-0.05724432  0.01825859  0.03333096 ... -0.06891205 -0.09311406
  -0.07138329]
 [-0.06942513  0.04892732  0.16244675 ... -0.14860214  0.09003196
  -0.08938965]
 [-0.00972351  0.01005133  0.05557884 ...  0.11449675 -0.03800575
  -0.02537755]]
indices: [0 1 2 3 4]
positional_info: [[-1.8820720e-02 -1.9741860e-01  4.0267250e-03 ... -4.3043736e-02
   2.8267192e-02  5.4490108e-02]
 [ 2.3959434e-02 -5.3792033e-02 -9.4878644e-02 ...  3.4170013e-02
   1.0171850e-02 -1.5572949e-04]
 [ 4.2160717e-03 -8.4763914e-02  5.4514930e-02 ...  1.9744711e-02
   1.9324856e-02 -2.1423856e-02]
 [-2.8336607e-04 -7.3802635e-02  1.0552647e-01 ...  1.0157347e-02
   1.7659478e-02 -7.0853787e-03]
 [ 7.6374277e-03 -2.5090257e-02  1.2695636e-01 ...  8.4643308e-03
   9.8542385e-03 -7.0117121e-03]]
x (embedded input): [[-0.08744361 -0.2

In [228]:
# Step 2: Transformer Block × 12 — 文脈理解と特徴変換を繰り返す
for block in blocks:
    y = block.ln_1(x)  # 正規化
    y = block.attn(y)  # 文脈理解
    x = x + y          # 残差接続（加算）
    z = block.ln_2(x)  # 正規化
    z = block.mlp(z)   # 特徴変換
    x = x + z          # 残差接続（加算）

print("x (after Transformer blocks):", x)
print("norm(x):", np.linalg.norm(x, axis=-1))

x (after Transformer blocks): [[  0.05047503  -0.17276424  -0.16896883 ...  -1.0095907    0.12802455
   -1.0686955 ]
 [-10.841122     3.4045794   -4.4607496  ...   2.7036042   -5.564179
   -1.0109675 ]
 [  3.229121     2.565983     1.1017268  ...   2.8642154   -1.2396975
   -0.40396878]
 [ -3.9180818   -0.24503696   0.4202696  ...   4.0475345   -5.226616
   -4.1138935 ]
 [ -5.9840055    0.8471148   -2.269939   ...   3.7883925   -0.632723
    1.0107007 ]]
norm(x): [379.15237 353.52875 434.4936  377.96616 429.59772]


In [229]:
# Step 3: 最終 LayerNorm — 出力前の正規化
x = ln_f(x)

print("x (after final LayerNorm):", x)
print("norm(x):", np.linalg.norm(x, axis=-1))

x (after final LayerNorm): [[-0.04703779 -0.03328462 -0.1625836  ... -0.13365638 -0.05713072
  -0.10593727]
 [-1.1896409   0.40008634 -0.7317769  ...  0.22102492 -0.5071472
  -0.06418581]
 [ 0.2946703   0.2673069   0.07318802 ...  0.19608943 -0.10776535
   0.00839415]
 [-0.4005001   0.01159246 -0.0094717  ...  0.31716618 -0.44648403
  -0.32983688]
 [-0.542677    0.10727708 -0.34962952 ...  0.25603366 -0.07385688
   0.10929476]]
norm(x): [ 78.14827 183.76552 182.14905 202.76149 255.3541 ]


In [230]:
# Step 4: LM Head — 全語彙の埋め込みベクトルとの内積（ロジット）を一括計算
logits = x @ wte.T

print("logits:", logits)

logits: [[ -36.287464  -35.01145   -38.0794   ...  -40.5164    -41.376015
   -34.919384]
 [ -75.10218   -75.64832   -82.68278  ...  -82.59613   -79.39135
   -76.26869 ]
 [ -80.096794  -78.6868    -81.23405  ...  -83.75481   -85.65412
   -79.804245]
 [ -86.00845   -86.46174   -91.018425 ...  -98.69116   -93.37344
   -87.92861 ]
 [-108.954216 -108.93266  -112.57931  ... -118.334496 -113.15045
  -110.37789 ]]


In [231]:
# Step 5: サンプリング — 確率分布から次のトークンを選択
probs = softmax(logits[-1])
next_id = np.random.choice(len(probs), p=probs)
next_token = tokenizer.decode([int(next_id)])

print("probs:", probs)
print("next_id:", next_id, "->", repr(next_token))
print("updated text:", repr(text + next_token))

probs: [1.4029463e-05 1.4335123e-05 3.7383828e-07 ... 1.1836884e-09 2.1117333e-07
 3.3786766e-06]
next_id: 1363 -> ' home'
updated text: 'The capital of France is home'


In [232]:
# 20 回サンプリングを繰り返す
for _ in range(20):
    next_id = np.random.choice(len(probs), p=probs)
    print("next_id:", next_id, "->", repr(tokenizer.decode([int(next_id)])))

next_id: 3284 -> ' Russia'
next_id: 2726 -> ' serious'
next_id: 783 -> ' now'
next_id: 287 -> ' in'
next_id: 783 -> ' now'
next_id: 706 -> ' after'
next_id: 262 -> ' the'
next_id: 319 -> ' on'
next_id: 257 -> ' a'
next_id: 3576 -> ' London'
next_id: 4881 -> ' France'
next_id: 2407 -> ' quite'
next_id: 783 -> ' now'
next_id: 1088 -> ' around'
next_id: 3729 -> ' certainly'
next_id: 262 -> ' the'
next_id: 4881 -> ' France'
next_id: 262 -> ' the'
next_id: 281 -> ' an'
next_id: 3706 -> ' named'


In [233]:
# 最終 LayerNorm 後のベクトル（LM Head の節で再利用）
hidden = x

# トークナイザー

## BPE（Byte Pair Encoding）

GPT-2 は「頻出するバイト列の塊」を学習によってトークンとして定義する BPE を採用しています。

処理の流れ：

1. **事前分割**: 正規表現でテキストを単語・記号に分割（`'Hello, world!'` → `['Hello', ',', ' world', '!']`）
2. **バイトマッピング**: UTF-8 バイト列を表示可能な Unicode 文字に 1:1 変換（空白 → `Ġ` など）
3. **BPE マージ**: `merges.txt` のルールに従い、隣り合うペアを優先度順に結合。rank は `merges.txt` での順位で、小さいほど学習データ中で頻出したペアであり優先して結合される

```text
"Hello" のマージ過程:
初期: ['H', 'e', 'l', 'l', 'o']
step1: ['l', 'l'] (rank=41)       → ['H', 'e', 'll', 'o']
step2: ['e', 'll'] (rank=439)     → ['H', 'ell', 'o']
step3: ['ell', 'o'] (rank=10853)  → ['H', 'ello']
step4: ['H', 'ello'] (rank=15240) → ['Hello']
```

4. **ID マッピング**: `vocab.json` でトークン文字列を ID に変換（`'Hello'` → ID 15496）

BPE はバイト単位に分解するため未知語が発生しませんが、日本語など分かち書きしない言語では非効率になります。

In [234]:
# 確認: merges.txt の rank と、トークナイザーの結果
for pair in [("l", "l"), ("e", "ll"), ("ell", "o"), ("H", "ello")]:
    print(pair, tokenizer.bpe_ranks[pair])

print(repr("hello"), "->", [(tokenizer.decode([i]), i) for i in encoded])

('l', 'l') 41
('e', 'll') 439
('ell', 'o') 10853
('H', 'ello') 15240
'hello' -> [('Hello', 15496)]


## SentencePiece（Unigram モデル）

`rinna/japanese-gpt2-small` は SentencePiece の **Unigram モデル**を使用します。

Unigram モデルは、語彙内の各ピース（部分文字列）に「出現確率の対数」をスコアとして持たせ、テキストを「スコアの合計が最大になるピース列」に分割します。確率の積は対数の和に変換されるため、アンダーフローを避けつつ最大化できます。

```text
log P(▁, 日本語) = log P(▁) + log P(日本語)
                 = (-3.5238) + (-9.9070)
                 = -13.4308
```

最適な分割の探索には **Viterbi アルゴリズム**（動的計画法）を用います。「位置 `i` までの最適分割が決まれば、それ以降はその結果だけを引き継げばよい」という最適部分構造を利用して、左から右へ効率的に解きます。

`▁日本語` を例にすると：

```text
best[0] = (  0.0   , -1, None)     ← 起点
best[1] = ( -3.5238,  0, "▁")     ← "▁" のスコア
best[4] = (-13.4308,  1, "日本語") ← "▁" + "日本語" が最適
バックトラック: best[4] → best[1] → best[0] → ["▁", "日本語"]
```

In [235]:
# 確認: スコアと分割結果
from my_gpt2.spiece import SentencePieceTokenizer

sp = SentencePieceTokenizer("rinna/japanese-gpt2-small")
print(sp._piece_to_score["▁"], sp._piece_to_score["日本語"])
print([(i, sp._id_to_piece[i]) for i in sp.encode("日本語")])

-3.523782253265381 -9.907026290893555
[(9, '▁'), (2481, '日本語')]


## BPE と Unigram の比較

両者の主な違いをまとめます。

| 観点 | BPE | Unigram |
|---|---|---|
| アルゴリズム | 最頻ペアを繰り返し結合 | 各ピースに対数確率スコアを持つ言語モデル |
| 分割方法 | 決定論的（マージ順に従う） | 最尤分割（Viterbi で最高スコアを探す） |
| ファイル形式 | `vocab.json` + `merges.txt` | `spiece.model`（Protocol Buffers） |
| 単語境界 | スペースをバイトとして埋め込み | `▁`（U+2581）マーカーで表現 |

# Embedding

トークン ID（整数）をベクトル（数値のリスト）に変換するステップです。GPT-2 では「単語の意味」と「位置情報」の 2 つのベクトルを足し合わせます。

- **WTE（Word Token Embedding）**: 語彙中の各トークンに対応する埋め込みベクトルを格納した行列。トークン ID で行を取り出すと、そのトークンの意味ベクトルが得られます。形状 (50257, 768)
- **WPE（Word Position Embedding）**: 位置を区別するためのベクトルの行列。Attention は入力の順序に依存しないため、位置情報を明示的に与えなければ "I love you" と "you love I" を区別できません。形状 (1024, 768)

In [236]:
x = wte[input_ids] + wpe[np.arange(len(input_ids))]

# 確認
print(len(input_ids), wte.shape, wpe.shape, x.shape)

5 (50257, 768) (1024, 768) (5, 768)


連結（concatenate）ではなく加算にすることで次元を維持し、パラメータ数を抑えています。加算しても学習によって位置と意味が自然に分離されることが知られています。

# Transformer Block

Transformer Block は LayerNorm・Attention・MLP・残差接続で構成されます。GPT-2 ではこれを 12 個積み重ねます。

```text
Input────┐
  │      ↓
  │  LayerNorm
  │      ↓
  │  Attention
  ↓      │
 (+)←────┘ 残差接続
  ├──────┐
  │      ↓
  │  LayerNorm
  │      ↓
  │     MLP
  ↓      │
 (+)←────┘ 残差接続
  ↓
Output
```

In [237]:
class TransformerBlock:
    def __call__(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp (self.ln_2(x))
        return x

# 確認：ランダムなベクトルを通す
input_vector = np.random.default_rng(0).standard_normal(
    (5, 768), dtype=np.float32
)
tb = TransformerBlock()
tb.__dict__.update(vars(blocks[0]))  # モデルの一部を読み込み
print("入力:", input_vector)
print("出力:", tb(input_vector))

入力: [[ 1.117622   -1.3871249  -0.4265716  ... -1.2837367   1.8048477
  -0.63249826]
 [-1.9112012  -0.9320151   3.0066628  ...  0.74511635 -0.25204748
   0.03077996]
 [ 0.991117    0.18805751 -2.1592162  ... -1.4761438  -0.9891986
   0.6017499 ]
 [-2.3152578  -0.8207919  -0.00496485 ... -0.7510275  -0.8909614
  -0.72919315]
 [-0.9561185  -0.33976084 -0.5302373  ... -0.5043844  -1.2192814
   0.6867705 ]]
出力: [[ 10.925539    -7.940941     9.268362   ...  -2.6521013   -1.0350518
   16.223198  ]
 [  8.24084     -9.016823     9.394821   ...  12.344389     1.4430909
    8.572675  ]
 [  3.1729493  -11.184773     0.55789924 ... -10.40501     -3.4550288
    1.8799427 ]
 [  0.70945024   1.8498049    0.22118524 ...   8.026867     6.813503
   13.028902  ]
 [  9.754632    -3.6678193   -9.843023   ...  -2.448443    11.058816
   18.826878  ]]


各層は大まかに異なる抽象度の特徴を担っていると考えられています。

- **浅い層（0〜3）**: 文法構造、品詞パターン、局所的な共起関係
- **中間層（4〜8）**: 意味的な関係、エンティティの同定、文脈の統合
- **深い層（9〜11）**: タスク固有の判断、最終的な予測に向けた情報の絞り込み

ただし明確に分かれるわけではなく、複数の層が協調して機能を実現しています。

## LayerNorm

残差接続によって各層の出力が積み重なると、ベクトルのスケールはトークンごと・層ごとにばらつきます。このばらつきを放置すると、Attention のスコア計算（内積）が特定の次元に支配されやすくなり、計算が不安定になります。LayerNorm は各処理の直前でスケールを揃えることで、この問題を防ぎます。

具体的には、各トークンのベクトル（768 次元）を**平均 0・分散 1** に正規化してから、学習済みパラメーター γ（スケール）と β（シフト）を適用します。

In [238]:
class LayerNorm:
    def __call__(self, x, eps=1e-5):
        mean = np.mean(x, axis=-1, keepdims=True)
        variance = np.var(x, axis=-1, keepdims=True)
        x_norm = (x - mean) / np.sqrt(variance + eps)
        return self.g * x_norm + self.b  # γ でスケール、β でシフト

# 確認：γ=1, β=0 なら正規化のみ。平均 0・分散 1 になる
ln = LayerNorm()
ln.g, ln.b = 1.0, 0.0
y = ln(x)
print(y.mean(axis=-1).round(3), y.var(axis=-1).round(3))

[-0.  0. -0. -0.  0.] [1. 1. 1. 1. 1.]


正規化で全次元を同じスケールにした後、γ/β で必要なスケールの違いを復元します。GPT-2 全体で 25 回使用されます（12 ブロック × 2 + 最終 ln_f × 1）。

## Attention

各トークンが他のトークンとの関係性を計算し、文脈を取り込む処理です。Transformer の原論文のタイトルは "Attention Is All You Need" であり、Attention こそが Transformer の中核です。

原論文では Encoder（双方向に参照）と Decoder（過去のみ参照）の 2 つが導入されましたが、GPT-2 は Decoder のみを使用します。Decoder の Attention は **Self-Attention** と呼ばれ、同じシーケンス内のトークン同士の関係を計算します。

### Q, K, V への変換

関連するトークンを探して情報を集める処理は、検索の比喩で 3 つの役割に分けられます。

- **Q (Query)**: 「何を探しているか」（検索クエリ）
- **K (Key)**: 「自分は何を持っているか」（検索インデックス）
- **V (Value)**: 「実際の情報」（検索結果）

入力ベクトルをそのまま使うと検索する側とされる側を区別できないため、異なる投影行列で変換します。768 次元を 12 個のヘッドに分割（各 64 次元）し、ヘッドごとに独立して Attention を計算します（**Multi-Head Attention**）。複数のヘッドがそれぞれ異なる観点で文脈を捉えることで、多面的な情報を取り込めます。

`seq_len` を入力のトークン列長とします。

In [239]:
# 以降の引用コードは Attention クラスの中身なので、self を用意して動かす
self = blocks[0].attn
x = blocks[0].ln_1(wte[input_ids] + wpe[np.arange(len(input_ids))])  # Attention への入力（LayerNorm 後）
seq_len, embed_dim = x.shape

In [240]:
# (seq_len, 768) → (seq_len, 2304)
qkv = x @ self.w_qkv + self.b_qkv
# (seq_len, 2304) → (seq_len, 768) × 3 分割
q, k, v = np.split(qkv, 3, axis=-1)
# 各 (seq_len, 768) → (seq_len, 12, 64) → (12, seq_len, 64)
q = q.reshape(seq_len, 12, 64).transpose(1, 0, 2)
k = k.reshape(seq_len, 12, 64).transpose(1, 0, 2)
v = v.reshape(seq_len, 12, 64).transpose(1, 0, 2)

# 確認
print(qkv.shape, q.shape)

(5, 2304) (12, 5, 64)


ヘッド当たりの次元数 64 を `d_k` とします。

In [241]:
d_k = q.shape[-1]
print(d_k)

64


### スコア計算

Q と K の内積で関連度を計算します。内積はベクトルが同じ方向を向いているほど大きくなるため、関連性の高いトークン同士ほどスコアが高くなります。次元数が大きいと内積の値が過大になるため、$\sqrt{d_k}$ でスケーリングします。

In [242]:
scores = q @ k.transpose(0, 2, 1) / math.sqrt(d_k)

# 確認
print(scores.shape)

(12, 5, 5)


> `np.sqrt(d_k)` は `np.float64` のスカラーを返すため、float32 の配列を割ると結果が float64 に格上げされ、以降の計算が遅くなります。そこで Python 標準の `math.sqrt` を使っています（GELU の定数も同様）。

### 因果マスキングと Softmax

GPT-2 は「次の単語を予測する」モデルのため、未来のトークンを参照できないよう上三角部分を $-10^{10}$ で上書きします。

スコアを確率分布に変換するため **Softmax** を適用します。Softmax は各スコアの指数 $\exp(x)$ を取り、合計で割ることで全体の和が 1 になるよう正規化する関数です。最もスコアが高い要素に高い確率が割り当てられます。マスクされた位置は $-10^{10}$ なので、$\exp(-10^{10}) \approx 0$ となり実質的に無視されます。

In [243]:
def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=-1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=-1, keepdims=True)

mask = np.tril(np.ones((seq_len, seq_len)))
scores = np.where(mask == 0, -1e10, scores)
probs = softmax(scores)  # 注目度行列

# 確認
print(probs[0].round(2))  # ヘッド 0 の注目度（上三角は 0）
print(probs[0].sum(axis=-1))  # 各行の合計は 1

[[1.   0.   0.   0.   0.  ]
 [0.85 0.15 0.   0.   0.  ]
 [0.7  0.22 0.07 0.   0.  ]
 [0.61 0.18 0.14 0.07 0.  ]
 [0.61 0.15 0.06 0.09 0.08]]
[1. 1. 1. 1. 1.]


`exp(x)` は値が大きいとオーバーフローするため、最大値を引いてから計算しています。最大値を引いても分子・分母に同じ定数がかかるため結果は変わりません。

### Value の重み付き平均

注目度を重みとして V の加重平均を取り、文脈を反映したベクトルを得ます。注目度の高いトークンほど多くの情報が流れ込みます。

In [244]:
out = probs @ v

# 確認
print(out.shape)

(12, 5, 64)


### 出力射影

最後に 12 ヘッドの結果を結合して 768 次元に戻し、出力射影を適用します。

In [245]:
out = out.transpose(1, 0, 2).reshape(seq_len, embed_dim)
out = out @ self.w_out + self.b_out

# 確認
print(np.allclose(out, self(x)))  # Attention 本体の出力と一致

True


## MLP

各トークンを**独立に**処理する特徴変換です。Attention が「トークン間の情報伝達」（横方向）なら、MLP は「トークン単体の意味の深掘り」（縦方向）です。Attention は情報を集約しますが、集めた情報を「解釈」する処理は行いません。MLP がその役割を担います。

768 次元のベクトルを一度 4 倍の 3072 次元に拡張し、非線形変換してから元の 768 次元に圧縮します。高次元空間で多くの特徴を同時に表現し、非線形変換で選別するという仕組みです。

In [246]:
class MLP:
    def __call__(self, x):
        i = x @ self.w_fc + self.b_fc         # 768 → 3072（4倍に拡張）
        a = gelu(i)                           # 非線形変換（活性化関数）
        return a @ self.w_proj + self.b_proj  # 3072 → 768（元に圧縮）

# 確認
mlp = MLP()
mlp.__dict__.update(vars(blocks[0].mlp))
z = blocks[0].ln_2(wte[input_ids] + wpe[np.arange(len(input_ids))])
print(np.allclose(mlp(z), blocks[0].mlp(z)))

True


GELU は ReLU を滑らかにした活性化関数で、正の値はほぼそのまま通過し、負の値は大きく抑制されます。非線形変換がないと、どれだけ層を重ねても全体が単なる線形変換に潰れてしまうため、非線形性の導入が不可欠です。

## 残差接続

`x = x + f(x)` という形で各処理の出力を元の入力に加算することを**残差接続**と呼びます。層の出力で上書きするのではなく「変更分を加える」構造です。これにより勾配消失を防ぎ、深いモデルでも学習が安定します。

情報の流れとして見ると、`x` は情報の幹線（**残差ストリーム**）であり、Attention や MLP は幹線に情報を「追加」するだけです。初期の Embedding は最終層まで直接伝わります。

GPT-2 は処理の**前**に LayerNorm を置く **Pre-LayerNorm** を採用しています。残差接続のパスがモデル全体を貫通するため、信号が深層まで伝わりやすくなります。

## 文脈付き埋め込み

12 ブロックを通過すると、同じ単語でも文脈によって異なるベクトルに変化します。これを**文脈付き埋め込み** (Contextual Embedding) と呼びます。例えば "river bank" と "money bank" の "bank" は、Embedding 層では同一ですが、Attention によって周囲の単語の情報を取り込み、異なるベクトルへと分岐していきます。

因果マスクにより、各トークンは自分自身と過去のトークンだけを参照できるため、最後のトークンはそれより前のすべてのトークンの情報が集約されます。

# 出力とサンプリング

12 層の Transformer Block と最終 LayerNorm を通過したベクトルから、次の単語の確率分布を生成します。

## LM Head（Weight Tying）

通常、出力層には独立した重み行列を用意しますが、GPT-2 では入力の Embedding 行列（WTE）をそのまま出力層にも再利用します。この手法を **Weight Tying** と呼びます。「単語 A を表すベクトル」と「次の単語として A を予測するベクトル」は同じ意味空間にあるべきという考えに基づいており、GPT-2 の 124M パラメーターのうち約 30% を占める行列を共有することでメモリも節約できます。

In [247]:
x = hidden  # 最終 LayerNorm 後のベクトル

logits = x @ wte.T  # (seq_len, 768) @ (768, 50257) → (seq_len, 50257)

# 確認
# 最後のトークンの上位 5 候補
for t in np.argsort(logits[-1])[::-1][:5]:
    print(repr(tokenizer.decode([int(t)])), logits[-1, t])

' the' -100.2498
' now' -100.81753
' a' -100.8555
' France' -101.21015
' Paris' -101.2143


行列積によって各トークンのベクトルと全語彙の埋め込みベクトルとの内積を一括計算しています。内積はベクトルが同じ方向を向いているほど大きくなるため、モデルの最終出力に近い埋め込みを持つトークンほど高いスコアになります。この出力は確率分布の前段階となるスコアで、**ロジット**と呼ばれます。

## サンプリング手法

因果マスクにより最後のトークンにはそれ以前のすべてのトークンの情報が集約されているため、次の単語の予測には最終トークンのロジットだけを使用します。

最終トークンのロジットを Softmax で確率分布に変換した後、次のトークンを選びます。確率分布からどのようにトークンを選ぶかによって、生成されるテキストの性質が変わります。

- **貪欲法**: 最も確率が高いトークンを選択。決定論的だがループが発生しやすい
- **Temperature**: ロジットを温度 T で割ってから Softmax を適用し、分布の尖り具合を調整。低温（T < 1）で保守的、高温（T > 1）で多様な生成になる
- **Top-k**: 上位 k 個のトークン以外をマスクし、候補数を固定
- **Top-p（Nucleus）**: 累積確率が p に達するまでのトークンを候補にする。語彙の分布に応じて候補数が動的に変わるため、Top-k より適応的

## 自己回帰生成

GPT-2 は一度に 1 トークンずつ予測します。予測したトークンを入力に追加して再び推論するループを繰り返すことで文章を生成します。

In [248]:
input_ids = tokenizer.encode(text)
n_tokens_to_generate = 8

for _ in range(n_tokens_to_generate):
    logits = model(np.array(input_ids))
    next_token = int(np.argmax(logits[-1, :]))
    input_ids.append(next_token)

# 確認
print(repr(tokenizer.decode(input_ids)))

'The capital of France is the capital of the French Republic, and'


毎回入力全体をモデルに通し、最後のトークンの確率分布から次のトークンを選びます。なお、この素朴な実装では毎回全トークンを再計算しています。次のセクションで説明する KV キャッシュを使えば、新しいトークンの計算だけで済むようになります。

# KV キャッシュ

自己回帰生成では毎回全トークンを再計算しますが、Transformer Block の中で他のトークンを参照するのは Attention だけです。MLP・LayerNorm・Embedding・LM Head はすべてトークン単位の独立した処理です。さらに因果マスクにより過去のトークンの K, V は変わりません。そこで計算済みの K と V を保存しておき、新しいトークンの計算だけで済ませるのが **KV キャッシュ**です。

- **Prefill**: 全プロンプトを処理して K, V をキャッシュに保存
- **Incremental**: 新トークンのみ処理し、K, V をキャッシュに追加。新しいトークンの Q とキャッシュ全体の K でスコアを計算

Attention の計算量は Q の長さ × K の長さに比例します。プロンプト 4 トークンから 3 トークンを生成する場合の計算量を比較すると：

```text
キャッシュなし（毎回全トークンを処理）:
  4×4 + 5×5 + 6×6 + 7×7 = 126

キャッシュあり（Prefill + 新トークンのみ処理）:
  4×4 + 1×5 + 1×6 + 1×7 =  34
```

シーケンスが長くなるほど差は広がります。

In [249]:
# 確認: KV キャッシュありでも結果は同じ。2 回目以降は新しいトークンだけを処理する
cached_ids = tokenizer.encode(text)
kv_cache = None  # Prefill 前は None
inputs = np.array(cached_ids)
for _ in range(n_tokens_to_generate):
    logits, kv_cache = model(inputs, kv_cache=kv_cache)
    next_token = int(np.argmax(logits[-1, :]))
    cached_ids.append(next_token)
    inputs = np.array([next_token])  # 新トークンのみ
print(cached_ids == input_ids)  # キャッシュなしの結果と一致

True


# GPT-2 から現代の LLM へ

GPT-2 のアーキテクチャは現代の LLM の基本形です。本質的な違いはパラメータ数と学習データの規模であり、基本構造は驚くほど変わっていません。

| | GPT-2 | 現代の LLM |
|---|---|---|
| パラメータ数 | 124M | 数十B〜数百B |
| 層数 | 12 | 数十〜百以上 |
| コンテキスト長 | 1024 | 100K〜2M |
| 位置埋め込み | 学習済み絶対位置 | RoPE 等の相対位置 |
| 正規化 | LayerNorm | RMSNorm |
| 活性化関数 | GELU | SwiGLU |
| Attention | 標準 MHA | GQA / MLA |

構造の違いは効率やスケーラビリティの改善であり、「残差ストリーム上で Attention と MLP を繰り返す」という根本的な設計は共通しています。GPT-2 で理解した原理は、そのまま現代の LLM にも応用できます。